# Tentativo 19 · 6. La valutazione

Tutte le misure si ricalcolano dalle uscite per articolo dei notebook 4 e 5 (`previsioni/`), per testata. Il notebook
salva le tabelle in `Results/`; il notebook 7 le mostra in una pagina.

**Le due filosofie sullo stesso terreno** (`metriche.py`):

| | spazio dei **livelli** | spazio del **voto** |
|---|---|---|
| classificatore | livello votato dagli articoli | voto atteso: probabilità × voto medio del livello nel training |
| regressore | confini NewsGuard · quote del training (16) · voto degli articoli (17) | media prevista · media riadattata ai quantili del training |
| nessun testo | classe più frequente nel training | media del training |

Nello spazio dei livelli: F1, accuratezza, **tolerant** (±1 livello), kappa, Spearman… Nello spazio del voto: MAE,
Spearman, R², **entro 10 e 20 punti** (la tolerant del voto)… Ogni trasformazione si calcola una volta; il bootstrap
ricampiona le testate (2.000 volte, seme fisso).

In [ ]:
NOMI = "TF-IDF,Qwen3 8B,Mistral 24B,ModernBERT"
B = 2000

In [ ]:
import warnings

import numpy as np
import pandas as pd

import metriche as Me
import modelli as Mo

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 300)
OUT = Mo.Path("Results"); OUT.mkdir(exist_ok=True)
art, t = Mo.leggi()
print(Mo.ambiente())

## 1. Quali modelli hanno tutti e 5 i fold

In [ ]:
PA = {}
for nome in NOMI.split(","):
    for filosofia, compito in [("classificatore", "classificazione"), ("regressore", "regressione")]:
        pa = Mo.previsioni(compito, nome)
        completo = pa is not None and pa.domain.nunique() == len(t)
        print(f"{nome:12s} {filosofia:15s} {'pronto' if completo else 'mancano dei fold'}")
        if completo:
            PA[(nome, filosofia)] = pa
NOMI = [n for n in NOMI.split(",") if (n, "classificatore") in PA or (n, "regressore") in PA]

## 2. Le viste di ogni modello e le loro misure (per testata)

In [ ]:
VISTE = {}                                                  # (modello, filosofia, vista) -> (spazio, vero, previsto)
for (nome, filosofia), pa in PA.items():
    if filosofia == "classificatore":
        viste = Me.viste_classificatore(Me.testate_classificatore(pa, t), t)
    else:
        viste = Me.viste_regressore(Me.testate_regressore(pa, t), pa, art)
    VISTE.update({(nome, filosofia, v): x for v, x in viste.items()})
VISTE.update({("nessun testo", "riferimento", v): x for v, x in Me.viste_riferimento(t).items()})
tutte = pd.concat([Me.misure_viste({k[2]: x}).assign(modello=k[0], filosofia=k[1]) for k, x in VISTE.items()], ignore_index=True)
tutte = tutte[["modello", "filosofia", "vista", "spazio"] + [c for c in tutte.columns if c not in ("modello", "filosofia", "vista", "spazio")]]
tutte.to_csv(OUT / "tutte_le_misure.csv", index=False)
print(f"{len(tutte)} viste, {tutte.shape[1] - 4} misure: Results/tutte_le_misure.csv")

**Spazio dei livelli.**

In [ ]:
colonne_l = ["F1 macro", "accuratezza", "accuratezza bilanciata", "tolerant (±1 livello)", "errore medio in livelli",
            "kappa quadratica", "Spearman", "affidabile: accuratezza"] + [f"F1 {l}" for l in Mo.LIVELLI]
tutte[tutte.spazio == "livelli"].set_index(["modello", "filosofia", "vista"])[colonne_l].round(3)

**Spazio del voto.**

In [ ]:
colonne_v = ["MAE", "RMSE", "R²", "Spearman", "Pearson", "CCC di Lin", "entro 10 punti", "entro 20 punti",
            "distorsione (previsto − vero)", "affidabile: accuratezza", "affidabile: AUC"]
tutte[tutte.spazio == "voto"].set_index(["modello", "filosofia", "vista"])[colonne_v].round(3)

## 3. Le misure con le probabilità del classificatore, e le misure per articolo

In [ ]:
prob = pd.DataFrame({nome: Me.misure_probabilita(Me.testate_classificatore(PA[(nome, "classificatore")], t))
                     for nome in NOMI if (nome, "classificatore") in PA}).T
prob.to_csv(OUT / "misure_probabilita.csv")
righe = []
for (nome, filosofia), pa in PA.items():                     # ogni articolo da solo, contro il valore della sua testata
    a = pa.merge(art[["article_id", "label", "score"]], on="article_id")
    m = Me.misure_livelli(a.label, a.previsione) if filosofia == "classificatore" else Me.misure_voto(a.score, a.previsione)
    righe.append({"modello": nome, "filosofia": filosofia, **m})
per_articolo = pd.DataFrame(righe); per_articolo.to_csv(OUT / "misure_per_articolo.csv", index=False)
display(prob.round(3))
per_articolo.round(3)

## 4. Intervalli al 95% delle misure principali

In [ ]:
CHIAVI = {"livelli": ["F1 macro", "tolerant (±1 livello)", "kappa quadratica", "Spearman", "affidabile: accuratezza"],
          "voto": ["MAE", "Spearman", "entro 10 punti", "entro 20 punti", "affidabile: accuratezza"]}
righe = []
for (nome, filosofia, vista), (spazio, y, p) in VISTE.items():
    ic = Me.bootstrap(y, p, spazio, CHIAVI[spazio], B=B)
    m = Me.MISURA[spazio](y, p)
    righe += [{"modello": nome, "filosofia": filosofia, "vista": vista, "spazio": spazio, "misura": c,
               "valore": m[c], "da": ic[c][0], "a": ic[c][1]} for c in CHIAVI[spazio]]
intervalli = pd.DataFrame(righe); intervalli.to_csv(OUT / "intervalli.csv", index=False)
print(f"{len(intervalli)} intervalli: Results/intervalli.csv")

## 5. Classificatore contro regressore, a coppie
Per ogni modello, le stesse testate, la stessa misura: **regressore − classificatore**. Nello spazio dei livelli il
classificatore usa il suo livello votato e il regressore le tre trasformazioni; nello spazio del voto il classificatore
usa il voto atteso e il regressore la media, così com'è o riadattata. Se l'intervallo non contiene lo 0, la differenza
non è caso. Per misure in cui meno è meglio (MAE, errore in livelli) una differenza negativa è a favore del regressore.

In [ ]:
CONFRONTI = {"livelli": ("livello votato dagli articoli", ["livelli: confini NewsGuard", "livelli: quote del training",
                                                              "livelli: voto degli articoli"], CHIAVI["livelli"]),
             "voto": ("voto atteso dalle probabilità", ["media prevista", "media riadattata ai quantili del training"], CHIAVI["voto"])}
righe = []
for nome in NOMI:
    if (nome, "classificatore") not in PA or (nome, "regressore") not in PA:
        continue
    for spazio, (vista_c, viste_r, chiavi) in CONFRONTI.items():
        for vista_r in viste_r:
            for c in chiavi:
                d, (a, b) = Me.bootstrap_differenza(VISTE[(nome, "regressore", vista_r)], VISTE[(nome, "classificatore", vista_c)], c, B=B)
                righe.append({"modello": nome, "spazio": spazio, "regressore": vista_r, "classificatore": vista_c, "misura": c,
                              "differenza": d, "da": a, "a": b, "non è caso": not (a <= 0 <= b)})
coppie = pd.DataFrame(righe); coppie.to_csv(OUT / "classificatore_contro_regressore.csv", index=False)
coppie.round(3)